In [ ]:
import shutil
import random
from pathlib import Path

print("--- Initiating Train/Test Split ---")

# Define our current and new directories
images_train_dir = Path('../data/images/train')
labels_train_dir = Path('../data/labels/train')

images_test_dir = Path('../data/images/test')
labels_test_dir = Path('../data/labels/test')
images_test_dir.mkdir(parents=True, exist_ok=True)
labels_test_dir.mkdir(parents=True, exist_ok=True)

# Grab all current training images
images = list(images_train_dir.glob('*.jpg'))
random.seed(42) # Ensures we get the exact same split if we run it twice

# Calculate 15% to move to the test set
test_ratio = 0.15
test_imgs = random.sample(images, int(len(images) * test_ratio))

moved_count = 0
for img in test_imgs:
    lbl = labels_train_dir / (img.stem + '.txt')
    
    # Move the image
    shutil.move(str(img), str(images_test_dir / img.name))
    
    # Move the corresponding label
    if lbl.exists():
        shutil.move(str(lbl), str(labels_test_dir / lbl.name))
    moved_count += 1

print(f"SUCCESS! Moved {moved_count} images and labels into the new Test set.")

In [ ]:
import cv2
import numpy as np
import random
import shutil
from pathlib import Path

print("--- Initiating Low-Light Underwater Augmentation ---")

def add_backscatter(img):
    noise = np.random.poisson(8, img.shape).astype(np.float32)
    return np.clip(img.astype(np.float32) + noise, 0, 255).astype(np.uint8)

def add_turbidity(img):
    haze = np.full(img.shape, (60, 50, 20), dtype=np.uint8)
    alpha = random.uniform(0.2, 0.5)
    return cv2.addWeighted(img, 1 - alpha, haze, alpha, 0)

def darken(img):
    gamma = random.uniform(0.3, 0.7)
    table = (np.arange(256) / 255.0) ** (1.0 / gamma) * 255
    return cv2.LUT(img, table.astype(np.uint8))

def add_blur(img):
    k = random.choice([3, 5])
    return cv2.GaussianBlur(img, (k, k), 0)

# Setup augmented directories
train_imgs_dir = Path('../data/images/train')
train_lbls_dir = Path('../data/labels/train')
aug_imgs_dir = Path('../data/images/train_aug')
aug_lbls_dir = Path('../data/labels/train_aug')

aug_imgs_dir.mkdir(parents=True, exist_ok=True)
aug_lbls_dir.mkdir(parents=True, exist_ok=True)

def augment_image_and_label(img_path, n_versions=3):
    img = cv2.imread(str(img_path))
    lbl_path = train_lbls_dir / (img_path.stem + '.txt')

    # 1. Preserve the original clean image and label in the new folder
    shutil.copy(str(img_path), str(aug_imgs_dir / img_path.name))
    if lbl_path.exists():
        shutil.copy(str(lbl_path), str(aug_lbls_dir / lbl_path.name))

    # 2. Generate the murky/dark variations
    for i in range(n_versions):
        aug = img.copy()
        if random.random() > 0.4: aug = darken(aug)
        if random.random() > 0.5: aug = add_turbidity(aug)
        if random.random() > 0.5: aug = add_backscatter(aug)
        if random.random() > 0.4: aug = add_blur(aug)
        
        new_stem = f"{img_path.stem}_aug{i}"
        
        # Save augmented image
        out_img = aug_imgs_dir / f"{new_stem}{img_path.suffix}"
        cv2.imwrite(str(out_img), aug)
        
        # Copy the exact same label coordinates for the new augmented image
        if lbl_path.exists():
            out_lbl = aug_lbls_dir / f"{new_stem}.txt"
            shutil.copy(str(lbl_path), str(out_lbl))

# Run on all training images
images_to_process = list(train_imgs_dir.glob('*.jpg'))
print(f"Found {len(images_to_process)} clean training images. Generating underwater variations...")

for img in images_to_process:
    augment_image_and_label(img, n_versions=3)

print("SUCCESS! All images augmented and labels successfully duplicated.")

In [ ]:
import cv2
import numpy as np
import random
import shutil
from pathlib import Path

print("--- Initiating Low-Light Underwater Augmentation ---")

def add_backscatter(img):
    noise = np.random.poisson(8, img.shape).astype(np.float32)
    return np.clip(img.astype(np.float32) + noise, 0, 255).astype(np.uint8)

def add_turbidity(img):
    haze = np.full(img.shape, (60, 50, 20), dtype=np.uint8)
    alpha = random.uniform(0.2, 0.5)
    return cv2.addWeighted(img, 1 - alpha, haze, alpha, 0)

def darken(img):
    gamma = random.uniform(0.3, 0.7)
    table = (np.arange(256) / 255.0) ** (1.0 / gamma) * 255
    return cv2.LUT(img, table.astype(np.uint8))

def add_blur(img):
    k = random.choice([3, 5])
    return cv2.GaussianBlur(img, (k, k), 0)

# Setup augmented directories
train_imgs_dir = Path('../data/images/train')
train_lbls_dir = Path('../data/labels/train')
aug_imgs_dir = Path('../data/images/train_aug')
aug_lbls_dir = Path('../data/labels/train_aug')

aug_imgs_dir.mkdir(parents=True, exist_ok=True)
aug_lbls_dir.mkdir(parents=True, exist_ok=True)

def augment_image_and_label(img_path, n_versions=3):
    img = cv2.imread(str(img_path))
    lbl_path = train_lbls_dir / (img_path.stem + '.txt')

    # 1. Preserve the original clean image and label in the new folder
    shutil.copy(str(img_path), str(aug_imgs_dir / img_path.name))
    if lbl_path.exists():
        shutil.copy(str(lbl_path), str(aug_lbls_dir / lbl_path.name))

    # 2. Generate the murky/dark variations
    for i in range(n_versions):
        aug = img.copy()
        if random.random() > 0.4: aug = darken(aug)
        if random.random() > 0.5: aug = add_turbidity(aug)
        if random.random() > 0.5: aug = add_backscatter(aug)
        if random.random() > 0.4: aug = add_blur(aug)
        
        new_stem = f"{img_path.stem}_aug{i}"
        
        # Save augmented image
        out_img = aug_imgs_dir / f"{new_stem}{img_path.suffix}"
        cv2.imwrite(str(out_img), aug)
        
        # Copy the exact same label coordinates for the new augmented image
        if lbl_path.exists():
            out_lbl = aug_lbls_dir / f"{new_stem}.txt"
            shutil.copy(str(lbl_path), str(out_lbl))

# Run on all training images
images_to_process = list(train_imgs_dir.glob('*.jpg'))
print(f"Found {len(images_to_process)} clean training images. Generating underwater variations...")

for img in images_to_process:
    augment_image_and_label(img, n_versions=3)

print("SUCCESS! All images augmented and labels successfully duplicated.")

In [ ]:
import yaml
from pathlib import Path

yaml_path = Path('../data/dataset.yaml')

with open(yaml_path, 'r') as file:
    data = yaml.safe_load(file)

# Point YOLO to the new augmented training folder and the new test folder
data['train'] = 'images/train_aug'
data['val'] = 'images/val'
data['test'] = 'images/test' 

with open(yaml_path, 'w') as file:
    yaml.dump(data, file, sort_keys=False)

print("dataset.yaml updated successfully for the new Augmented pipeline!")